In [1]:
# Cell 1: Setup and Raw Data Generation
import os
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

# 1. Install necessary libraries (if not already installed)
# We use %%capture to keep the output clean, remove it if you want to see installation logs
get_ipython().system('pip install dvc mlflow pandas scikit-learn')

# 2. Initialize Git and DVC (DVC requires a git repo to track metadata)
if not os.path.exists('.git'):
    get_ipython().system('git init')
    print("Initialized Git repository.")

if not os.path.exists('.dvc'):
    get_ipython().system('dvc init')
    print("Initialized DVC.")

# 3. Configure DVC to not use a remote for now (to keep it simple locally first)
# If you want to use the Google Drive bonus immediately, let me know, 
# otherwise we will stick to local storage to ensure you finish the assignment first.
get_ipython().system('dvc config core.analytics false')

df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
df["target"] = df["label"].map({"ham":0, "spam":1})
df = df.drop(columns=["label"])
df.to_csv("raw_data.csv", index=False)

print("raw_data.csv created successfully.")

# 5. Track raw_data.csv with DVC
get_ipython().system('dvc add raw_data.csv')
get_ipython().system('git add raw_data.csv.dvc .gitignore')
get_ipython().system('git commit -m "Add raw data"')

print("\nSetup complete.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
Initialized DVC.
raw_data.csv created successfully.

To track the changes with git, run:

	git add .gitignore raw_data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


⠋ Checking graph



[master 4975f43] Add raw data
 1 file changed, 3 deletions(-)

Setup complete.


In [2]:
# Cell 2: First Split (Random Seed 42)
from sklearn.model_selection import train_test_split

# 1. Load Data
df = pd.read_csv('raw_data.csv')

# 2. Split Data (60% Train, 20% Val, 20% Test)
# First split: Train (60%) vs Temp (40%)
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42)
# Second split: Validation (50% of Temp -> 20% total) vs Test (50% of Temp -> 20% total)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# 3. Save Splits
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)
test_df.to_csv('test.csv', index=False)

# 4. Print Distribution
print("=== Distribution for Version 1 (Seed 42) ===")
print("Train Target Counts:\n", train_df['target'].value_counts())
print("\nValidation Target Counts:\n", val_df['target'].value_counts())
print("\nTest Target Counts:\n", test_df['target'].value_counts())

# 5. Track with DVC
get_ipython().system('dvc add train.csv validation.csv test.csv')
get_ipython().system('git add train.csv.dvc validation.csv.dvc test.csv.dvc .gitignore')
get_ipython().system('git commit -m "Add data split version 1 (seed 42)"')

# Tag this version so we can easily return to it later
get_ipython().system('git tag -a v1 -m "Version 1 data split"')

=== Distribution for Version 1 (Seed 42) ===
Train Target Counts:
 target
0    2887
1     456
Name: count, dtype: int64

Validation Target Counts:
 target
0    981
1    133
Name: count, dtype: int64

Test Target Counts:
 target
0    957
1    158
Name: count, dtype: int64

To track the changes with git, run:

	git add train.csv.dvc validation.csv.dvc .gitignore test.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


⠋ Checking graph



[master 5c016ee] Add data split version 1 (seed 42)
 1 file changed, 3 insertions(+)


fatal: tag 'v1' already exists


In [3]:
# Cell 3: Update Split (Version 2 - Random Seed 2026)

# 1. Load Data
df = pd.read_csv('raw_data.csv')

# 2. Split Data with DIFFERENT Seed (2026)
# First split: Train (60%) vs Temp (40%)
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=2026)
# Second split: Validation (50% of Temp -> 20% total) vs Test (50% of Temp -> 20% total)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=2026)

# 3. Save Splits (Overwriting previous files)
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)
test_df.to_csv('test.csv', index=False)

print("Files updated with new random seed (2026).")

# 4. Track changes with DVC
get_ipython().system('dvc add train.csv validation.csv test.csv')

# 5. Commit changes to Git
get_ipython().system('git add train.csv.dvc validation.csv.dvc test.csv.dvc')
get_ipython().system('git commit -m "Update data split version 2 (seed 2026)"')

# Tag this version
get_ipython().system('git tag -a v2 -m "Version 2 data split"')

print("Version 2 tracked and committed.")

Files updated with new random seed (2026).

To track the changes with git, run:

	git add test.csv.dvc train.csv.dvc validation.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


⠋ Checking graph



[master e5a9d31] Update data split version 2 (seed 2026)
 3 files changed, 6 insertions(+), 6 deletions(-)
Version 2 tracked and committed.


fatal: tag 'v2' already exists


In [4]:
# Cell 4: Checkout V1, Print, then Checkout V2, Print

import pandas as pd

def print_distribution(version_label):
    print(f"\n{'='*10} Target Distribution for {version_label} {'='*10}")
    for filename in ['train.csv', 'validation.csv', 'test.csv']:
        try:
            data = pd.read_csv(filename)
            counts = data['target'].value_counts().to_dict()
            print(f"{filename}: {counts}")
        except FileNotFoundError:
            print(f"{filename}: Not found")

# --- 1. Switch to Version 1 ---
print(">>> Switching to Version 1 (Seed 42)...")
# Revert the .dvc files to the state they were in at tag 'v1'
get_ipython().system('git checkout v1')
# Restore the actual data files based on those .dvc files
get_ipython().system('dvc checkout')

# Verify V1 Data
print_distribution("VERSION 1")

# --- 2. Switch back to Version 2 ---
print("\n>>> Switching to Version 2 (Seed 2026)...")
# Switch back to the master branch (latest version)
get_ipython().system('git checkout master')
# Restore the data for V2
get_ipython().system('dvc checkout')

# Verify V2 Data
print_distribution("VERSION 2")

>>> Switching to Version 1 (Seed 42)...
M	.dvc/config


Note: switching to 'v1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at d84f96b Add data split version 1 (seed 42)



========== Target Distribution for VERSION 1 ==========
train.csv: Not found
validation.csv: Not found
test.csv: Not found

>>> Switching to Version 2 (Seed 2026)...


ERROR: Checkout failed for following targets:
raw_data.csv
train.csv
test.csv
validation.csv
Is your cache up to date?
<https://error.dvc.org/missing-files>


M	.dvc/config


Previous HEAD position was d84f96b Add data split version 1 (seed 42)
Switched to branch 'master'


A       raw_data.csv
A       test.csv
A       train.csv
A       validation.csv

========== Target Distribution for VERSION 2 ==========
train.csv: {0: 2900, 1: 443}
validation.csv: {0: 949, 1: 165}
test.csv: {0: 976, 1: 139}
